<a href="https://colab.research.google.com/github/simon-clematide/il-reimplementation-demo/blob/main/notebooks/morphological_inflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Transducer: Morphological Inflection Tutorial

This tutorial demonstrates how to train and evaluate transition-based neural string transducers ([Makarov & Clematide, 2020](https://aclanthology.org/2020.sigmorphon-1.19)) using imitation learning for **morphological inflection generation**.

We use the official **CoNLL-SIGMORPHON 2017 Shared Task 1** benchmark (English, German, French, Italian) to explore:
1. **Low-resource learning:** 100 training examples.
2. **Medium-resource learning:** 1,000 training examples.
3. **Stochastic Edit Distance (SED):** Unsupervised minimal edit alignment.
4. **Action traces:** Inspecting step-by-step edit operations (`COPY`, `DELETE`, `INSERT`, `SUBSTITUTE`).

## 1. Environment Setup & Dependencies

Install the `neural_transducer` package (pinned to official commit) and required runtime dependencies.

In [ ]:
# Install neural_transducer from GitHub
!pip install -q git+https://github.com/simon-clematide/il-reimplementation.git@f71833192acd505bc1d3ba47d75798cfb778fc3d
!pip install -q huggingface_hub

import torch
import trans
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Data Acquisition (CoNLL-SIGMORPHON 2017)

Download the benchmark dataset directly from the pinned repository commit: `519bb1677a7c747b3863d37069ffd395062288d8`.

In [ ]:
import os
import urllib.request

REPO_BASE = "https://raw.githubusercontent.com/sigmorphon/conll2017/519bb1677a7c747b3863d37069ffd395062288d8"
os.makedirs("data", exist_ok=True)

# Download Italian splits (low, medium, dev, test)
splits = {
    "train_low": "all/task1/italian-train-low",
    "train_medium": "all/task1/italian-train-medium",
    "dev": "all/task1/italian-dev",
    "test": "answers/task1/italian-uncovered-test",
}

for split_name, remote_path in splits.items():
    local_path = f"data/ita_{split_name}.tsv"
    if not os.path.exists(local_path):
        url = f"{REPO_BASE}/{remote_path}"
        urllib.request.urlretrieve(url, local_path)
        print(f"Downloaded {local_path}")

# Peek at the training data
!head -n 5 data/ita_train_low.tsv

## 3. Inspecting Morphological Features & Samples

Notice that each sample consists of:
`lemma <TAB> inflected_form <TAB> unimorph_features`

In [ ]:
with open("data/ita_train_low.tsv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        lemma, form, feats = line.strip().split("\t")
        print(f"Lemma: {lemma:15s} -> Inflected: {form:15s} | Features: {feats}")

## 4. Fast Inference with Pre-trained Models

You can download and run inference immediately with our pre-trained Hugging Face bundles:

In [ ]:
from huggingface_hub import snapshot_download

# Download pre-trained Italian bundles from the Hub
bundle_dir = snapshot_download(
    repo_id="simon-clmtd/neural-transducer-inflection",
    allow_patterns="ita_1000/*",
    local_dir="pretrained",
)
print("Downloaded model bundle to:", bundle_dir)

## 5. Training Your Own Transducer (`trans-train`)

Now let's train a model from scratch on the low-resource dataset (100 examples) using `trans-train`.

Notice how fast the training runs:
- SED alignment is fitted first via Expectation-Maximization (EM).
- The neural transducer learns transition edit policies via imitation learning with greedy decoding.

In [ ]:
!trans-train \
    --train data/ita_train_low.tsv \
    --dev data/ita_dev.tsv \
    --test data/ita_test.tsv \
    --output models/ita_100_custom \
    --device cuda \
    --pytorch-seed 42 \
    --epochs 60 \
    --enc-type lstm \
    --enc-hidden-dim 200 \
    --enc-layers 1 \
    --enc-output-dropout 0.3 \
    --char-dim 100 \
    --feat-dim 100 \
    --action-dim 100 \
    --dec-hidden-dim 200 \
    --optimizer adamw \
    --lr 0.001 \
    --scheduler reduce_on_plateau \
    --factor 0.5 \
    --lrs-patience 4 \
    --patience 20 \
    --batch-size 8 \
    --beam-width 0

## 6. Evaluation Results

Check the test set evaluation results produced during training:

In [ ]:
!cat models/ita_100_custom/test_greedy.eval

## 7. Interactive Single-Sample Prediction

Let's inspect step-by-step predictions using the custom-trained model:

In [ ]:
import json
import argparse
from trans import vocabulary, sed, optimal_expert_substitutions, transducer

# Load trained artifacts
output_dir = "models/ita_100_custom"
with open(f"{output_dir}/best.model.json") as f:
    meta = json.load(f)

args = argparse.Namespace(**meta["args"])
args.device = "cuda" if torch.cuda.is_available() else "cpu"

vocab = vocabulary.FeatureVocabularies.from_pickle(f"{output_dir}/vocabulary.pkl")
aligner = sed.StochasticEditDistance.from_pickle(f"{output_dir}/sed.pkl")
expert = optimal_expert_substitutions.OptimalSubstitutionExpert(aligner)
model = transducer.Transducer(vocab, expert, args)
model.load_state_dict(torch.load(f"{output_dir}/best.model", map_location=args.device))
model.eval()

# Interactive prediction helper
def inflect(lemma, feats):
    in_tok = model.source_tokenizer.tokenize(lemma)
    enc_in = torch.tensor(vocab.encode_unseen_input(in_tok), device=args.device).unsqueeze(0)
    enc_f = torch.tensor(vocab.encode_unseen_features(feats), device=args.device).unsqueeze(0)
    with torch.no_grad():
        out = model.transduce([in_tok], enc_in, enc_f)
    print(f"{lemma} + [{feats}] => {out.output[0]} (score: {out.log_p:.3f})")

inflect("parlare", "V;IND;FUT;1;SG")
inflect("dormire", "V;IND;PRS;2;PL")
inflect("vedere", "V;IND;PST;3;SG")

## 8. Summary & Next Steps

- **Low vs. Medium:** Comparing 100 examples against 1,000 examples demonstrates substantial improvements on irregular forms and tense alternations.
- **Hugging Face Space:** Visit our interactive web demo at [huggingface.co/spaces/simon-clmtd/morphological-inflection](https://huggingface.co/spaces/simon-clmtd/morphological-inflection).